In [ ]:
# =========================
# 0) PACKAGES
# =========================
import sys, subprocess
def _pip_install(pkgs):
    try:
        print("[setup] installing:", " ".join(pkgs))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
    except Exception as e:
        print("[setup] pip install failed (will continue):", e)

_pip_install([
    "akshare>=1.13.93",
    "pandas>=2.2.2",
    "pyarrow>=16.1.0",
    "fastparquet>=2024.5.0",
    "SQLAlchemy>=2.0.30",
    "tushare>=1.2.89",
])

In [12]:
# =========================
# 1) IMPORT
# =========================

from __future__ import annotations

import sys, subprocess, os, random, time
import datetime as dt

from pathlib import Path
from typing import Callable, Dict, Optional, Sequence, Tuple, Type, Union

import pandas as pd
import numpy as np
import akshare as ak
from sqlalchemy import create_engine

import glob
import shutil

In [9]:
# =========================
# 2) CONFIG
# =========================
CONFIG = {
    "DATA_DIR": "data",
    "DB_URL":   "sqlite:///data/meta.db",
    "SAMPLE_N": 10,                 # None when stable
    "HIST_START": "2022-01-01",
    "HIST_END":   dt.date.today().strftime("%Y-%m-%d"),
    "BASE_SLEEP": 0.35,
    "INDEX_LIST": ["000300.SH", "000905.SH", "399006.SZ"],
    "RUNS": {
        "trading_calendar": True,
        "securities_list":  True,
        "index_members":    False,
        "daily_bars":       True,
        "corp_actions":     True,
        "standardized_daily_bars": True,
        "market_features": True,
    }
}


os.makedirs(CONFIG["DATA_DIR"], exist_ok=True)
engine = create_engine(CONFIG["DB_URL"], echo=False)

# =========================
# 3) UTILITIES
# =========================

# Reusable helper functions shared across the ETL pipeline.
# Designed to reduce duplication and standardize data handling.

def ensure_dirs():
    """
    Layered data directories.

    data/
        raw/
            market/daily_by_symbol/
            events/corp_actions_raw/
            reference/
        standardized/
            market/daily_bars/
            events/corp_actions/
            reference/
        feature/
        factor/
    """
    for p in [
        f'{CONFIG["DATA_DIR"]}/raw/market/daily_by_symbol',
        f'{CONFIG["DATA_DIR"]}/raw/events/corp_actions_raw',
        f'{CONFIG["DATA_DIR"]}/raw/reference',

        f'{CONFIG["DATA_DIR"]}/standardized/market/daily_bars',
        f'{CONFIG["DATA_DIR"]}/standardized/events/corp_actions',
        f'{CONFIG["DATA_DIR"]}/standardized/reference',

        f'{CONFIG["DATA_DIR"]}/feature',
        f'{CONFIG["DATA_DIR"]}/factor',
    ]:
        os.makedirs(p, exist_ok=True)

def save_parquet(df: pd.DataFrame, path: str):
    """
    Save DataFrame as parquet (efficient columnar format).

    Core idea:
    - Skip empty data
    - Ensure directory exists

    Typical usage:
    save_parquet(df, "data/equities/daily/000001.SZ.parquet")
    """
    if df is None or df.empty:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)

def upsert_sqlite(df: pd.DataFrame, table: str, keys: list):
    """
    Insert or update records in SQLite (simulate upsert).

    Core idea:
    - Merge old + new data
    - Deduplicate using primary keys

    Typical usage:
    upsert_sqlite(df, "securities", ["ts_code"])
    """
    if df is None or df.empty:
        return

    df = df.copy()

    try:
        old = pd.read_sql_table(table, con=engine)
        if old is not None and not old.empty:
            df = pd.concat([old, df], ignore_index=True)
        df = df.drop_duplicates(subset=keys, keep="last")
    except Exception:
        # Table does not exist yet
        pass

    df.to_sql(table, engine, if_exists="replace", index=False)

def ensure_dataset_registry_table():
    """
    Create dataset registry table if not exists.
    """
    sql = """
    CREATE TABLE IF NOT EXISTS dataset_registry (
        dataset_name TEXT PRIMARY KEY,
        layer TEXT,
        path TEXT,
        row_count INTEGER,
        partition_count INTEGER,
        last_updated TEXT,
        schema_version TEXT
    )
    """
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

def update_dataset_registry(
    dataset_name: str,
    layer: str,
    path: str,
    row_count: int,
    partition_count: int,
    schema_version: str = "v1"
):
    """
    Stable upsert for dataset_registry:
    delete same dataset_name first, then append one fresh row.
    """
    ensure_dataset_registry_table()

    rec = pd.DataFrame([{
        "dataset_name": dataset_name,
        "layer": layer,
        "path": path,
        "row_count": int(row_count),
        "partition_count": int(partition_count),
        "last_updated": dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "schema_version": schema_version,
    }])

    with engine.begin() as conn:
        # delete existing row for same dataset_name
        conn.exec_driver_sql(
            "DELETE FROM dataset_registry WHERE dataset_name = :name",
            {"name": dataset_name}
        )

    # append new row
    rec.to_sql("dataset_registry", engine, if_exists="append", index=False)

def ensure_pipeline_runs_table():
    sql = """
    CREATE TABLE IF NOT EXISTS pipeline_runs (
        run_id TEXT,
        step_name TEXT,
        start_time TEXT,
        end_time TEXT,
        status TEXT,
        message TEXT,
        rows INTEGER,
        duration_seconds REAL
    )
    """
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

def append_pipeline_run(
    run_id: str,
    step_name: str,
    start_time: str,
    end_time: str,
    status: str,
    message: str = "",
    rows: int | None = None,
    duration_seconds: float | None = None,
):
    ensure_pipeline_runs_table()

    rec = pd.DataFrame([{
        "run_id": run_id,
        "step_name": step_name,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "message": message,
        "rows": None if rows is None else int(rows),
        "duration_seconds": None if duration_seconds is None else float(duration_seconds),
    }])

    try:
        old = pd.read_sql_table("data_quality_reports", con=engine)
        if old is not None and not old.empty:
            issues = pd.concat([old, issues], ignore_index=True)
    except Exception:
        pass

    rec.to_sql("pipeline_runs", engine, if_exists="replace", index=False)

def ensure_data_quality_table():
    sql = """
    CREATE TABLE IF NOT EXISTS data_quality_reports (
        dataset_name TEXT,
        check_name TEXT,
        failed_rows INTEGER,
        detail TEXT,
        checked_at TEXT
    )
    """
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

def write_quality_report(dataset_name: str, issues: pd.DataFrame):
    ensure_data_quality_table()

    if issues is None or issues.empty:
        return

    issues = issues.copy()
    issues["dataset_name"] = dataset_name
    issues["checked_at"] = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        old = pd.read_sql_table("data_quality_reports", con=engine)
        if old is not None and not old.empty:
            issues = pd.concat([old, issues], ignore_index=True)
    except Exception:
        pass

    issues.to_sql("data_quality_reports", engine, if_exists="replace", index=False)

def safe_call(fn, *args, tries=3, sleep=0.8, jitter=0.5, **kwargs):
    for i in range(tries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if "No tables found" in msg:
                return None

            if i == tries - 1:
                print(f"[safe_call] {fn.__name__} failed: {e}")
                return None

            wait = sleep * (i + 1) + random.random() * jitter
            time.sleep(wait)

def infer_ts(code: str) -> str:
    """
    Convert stock code to Tushare format (CODE.EXCHANGE).

    Core idea:
    - Standardize code format across data sources

    Typical usage:
    infer_ts("600000") → "600000.SH"
    """
    code = str(code).strip()

    # Already in TS format
    if "." in code:
        return code

    if code.startswith("6"):
        return f"{code}.SH"
    if code.startswith(("0", "3")):
        return f"{code}.SZ"
    if code.startswith(("8", "4")):
        return f"{code}.BJ"

    return f"{code}.CN"

def _parquet_path_for_symbol(ts_code: str) -> str:
    return f'{CONFIG["DATA_DIR"]}/raw/market/daily_by_symbol/{ts_code.replace(".","_")}.parquet'


def _get_last_trade_date_from_parquet(path: str) -> str | None:
    """
    Return max trade_date (YYYY-MM-DD) from an existing parquet file.
    If file does not exist or unreadable, return None.
    """
    if not os.path.exists(path):
        return None
    try:
        old = pd.read_parquet(path, columns=["trade_date"])
        if old is None or old.empty:
            return None
        # trade_date is stored as string "YYYY-MM-DD" in your pipeline
        return str(old["trade_date"].max())
    except Exception as e:
        print(f"[daily_k] WARN: cannot read existing parquet {path}: {e}")
        return None

def _next_day(date_str: str) -> str:
    d = pd.to_datetime(date_str, errors="coerce")
    if pd.isna(d):
        return None
    return (d + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

def validate_daily_bars(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic quality checks for standardized daily bars.

    Returns:
        issue dataframe with columns:
        check_name, failed_rows, detail
    """
    issues = []

    if df is None or df.empty:
        issues.append({
            "check_name": "empty_dataframe",
            "failed_rows": 0,
            "detail": "input dataframe is empty"
        })
        return pd.DataFrame(issues)

    required = ["ts_code", "trade_date", "open", "high", "low", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        issues.append({
            "check_name": "missing_required_columns",
            "failed_rows": len(missing),
            "detail": ",".join(missing)
        })
        return pd.DataFrame(issues)

    tmp = df.copy()

    # Parse date
    parsed_date = pd.to_datetime(tmp["trade_date"], errors="coerce")
    bad_date = parsed_date.isna().sum()
    if bad_date > 0:
        issues.append({
            "check_name": "invalid_trade_date",
            "failed_rows": int(bad_date),
            "detail": "trade_date cannot be parsed"
        })

    # Coerce numeric
    for c in ["open", "high", "low", "close"]:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
        bad_num = tmp[c].isna().sum()
        if bad_num > 0:
            issues.append({
                "check_name": f"non_numeric_{c}",
                "failed_rows": int(bad_num),
                "detail": f"{c} contains non-numeric values"
            })

    if "vol" in tmp.columns:
        tmp["vol"] = pd.to_numeric(tmp["vol"], errors="coerce")
        bad_vol = tmp["vol"].isna().sum()
        if bad_vol > 0:
            issues.append({
                "check_name": "non_numeric_vol",
                "failed_rows": int(bad_vol),
                "detail": "vol contains non-numeric values"
            })

    # Key uniqueness
    dup = tmp.duplicated(subset=["ts_code", "trade_date"]).sum()
    if dup > 0:
        issues.append({
            "check_name": "duplicate_ts_code_trade_date",
            "failed_rows": int(dup),
            "detail": "duplicate primary key rows"
        })

    # OHLC bounds
    valid_ohlc_mask = (
        tmp["low"].notna() & tmp["high"].notna() &
        tmp["open"].notna() & tmp["close"].notna()
    )
    bad_ohlc = (
        (tmp.loc[valid_ohlc_mask, "low"] > tmp.loc[valid_ohlc_mask, "high"]) |
        (tmp.loc[valid_ohlc_mask, "open"] < tmp.loc[valid_ohlc_mask, "low"]) |
        (tmp.loc[valid_ohlc_mask, "open"] > tmp.loc[valid_ohlc_mask, "high"]) |
        (tmp.loc[valid_ohlc_mask, "close"] < tmp.loc[valid_ohlc_mask, "low"]) |
        (tmp.loc[valid_ohlc_mask, "close"] > tmp.loc[valid_ohlc_mask, "high"])
    ).sum()
    if bad_ohlc > 0:
        issues.append({
            "check_name": "invalid_ohlc_bounds",
            "failed_rows": int(bad_ohlc),
            "detail": "low <= open/close <= high violated"
        })

    # Negative prices
    for c in ["open", "high", "low", "close"]:
        neg = (tmp[c] < 0).sum()
        if neg > 0:
            issues.append({
                "check_name": f"negative_{c}",
                "failed_rows": int(neg),
                "detail": f"{c} contains negative values"
            })

    # Negative volume
    if "vol" in tmp.columns:
        neg_vol = (tmp["vol"] < 0).sum()
        if neg_vol > 0:
            issues.append({
                "check_name": "negative_vol",
                "failed_rows": int(neg_vol),
                "detail": "vol contains negative values"
            })

    return pd.DataFrame(issues)

def validate_market_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic quality checks for market feature dataset.
    """
    issues = []

    if df is None or df.empty:
        issues.append({
            "check_name": "empty_dataframe",
            "failed_rows": 0,
            "detail": "feature dataframe is empty"
        })
        return pd.DataFrame(issues)

    required = [
    "ts_code", "trade_date", "close",
    "ret_1d", "ret_5d", "ret_20d",
    "ma_5", "ma_20", "ma_ratio_20",
    "volatility_20d"
]
    missing = [c for c in required if c not in df.columns]
    if missing:
        issues.append({
            "check_name": "missing_required_columns",
            "failed_rows": len(missing),
            "detail": ",".join(missing)
        })
        return pd.DataFrame(issues)

    dup = df.duplicated(subset=["ts_code", "trade_date"]).sum()
    if dup > 0:
        issues.append({
            "check_name": "duplicate_ts_code_trade_date",
            "failed_rows": int(dup),
            "detail": "duplicate feature keys"
        })

    feature_cols = ["ret_1d", "ret_5d", "ma_5", "ma_20", "volatility_20d", "ret_20d", "ma_ratio_20"]
    for c in feature_cols:
        s = pd.to_numeric(df[c], errors="coerce")

        inf_count = np.isinf(s).sum()
        if inf_count > 0:
            issues.append({
                "check_name": f"infinite_{c}",
                "failed_rows": int(inf_count),
                "detail": f"{c} contains inf or -inf"
            })

        if s.notna().sum() == 0:
            issues.append({
                "check_name": f"all_null_{c}",
                "failed_rows": len(df),
                "detail": f"{c} is entirely null"
            })

    return pd.DataFrame(issues)

# =========================
# 4) EXTRACT
# =========================

def fetch_trading_calendar():
    """
    Fetch trading calendar and store in SQLite.

    Core idea:
    - Standardize date column to 'trade_date'
    - Store as YYYY-MM-DD format

    Typical usage:
    fetch_trading_calendar()
    """
    df = safe_call(ak.tool_trade_date_hist_sina)

    if df is None or df.empty:
        print("[calendar] WARN: empty")
        return

    # Normalize column name to 'trade_date'
    if "trade_date" not in df.columns:
        for c in df.columns:
            if "date" in c.lower():
                df = df.rename(columns={c: "trade_date"})
                break

    df = df[["trade_date"]].copy()
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.strftime("%Y-%m-%d")
    df["exchange"] = "CN"

    upsert_sqlite(df, "trading_calendar", ["trade_date"])
    print(f"[calendar] {len(df)} rows")

def fetch_securities_list():
    """
    Fetch A-share securities list with multi-endpoint fallback (AkShare only),
    then TuShare as optional fallback if token exists.

    Output schema: ts_code, name
    """
    df_raw, source = None, None

    # ---- AkShare candidates (try several endpoints) ----
    ak_candidates = [
        ("ak_stock_zh_a_spot_em", getattr(ak, "stock_zh_a_spot_em", None), {}),
        ("ak_stock_zh_a_spot",    getattr(ak, "stock_zh_a_spot", None), {}),
        ("ak_stock_zh_a_spot_hs", getattr(ak, "stock_zh_a_spot_hs", None), {}),
    ]

    for name, fn, kwargs in ak_candidates:
        if fn is None:
            continue
        df_try = safe_call(fn, tries=6, sleep=1.2, jitter=1.2, **kwargs)
        if df_try is not None and not df_try.empty:
            df_raw, source = df_try, name
            print(f"[securities] fetched from AkShare ({source}): {len(df_raw)} rows")
            break

    # ---- TuShare fallback (optional) ----
    if (df_raw is None or df_raw.empty):
        token = os.getenv("TUSHARE_TOKEN", "").strip()
        if token:
            try:
                import tushare as ts
                pro = ts.pro_api(token)
                df_ts = pro.stock_basic(exchange="", list_status="L", fields="ts_code,name,symbol")
                if df_ts is not None and not df_ts.empty:
                    df_raw, source = df_ts, "tushare_basic"
                    print(f"[securities] fetched from TuShare ({source}): {len(df_raw)} rows")
            except Exception as e:
                print(f"[securities] WARN: TuShare fallback failed: {e}")
        else:
            print("[securities] WARN: TUSHARE_TOKEN not set")

    if df_raw is None or df_raw.empty:
        print("[securities] ERROR: failed to fetch securities list")
        return

    # ---- Normalize columns ----
    if source.startswith("ak_"):
        code_col = next((c for c in df_raw.columns if "代码" in str(c)), None)
        name_col = next((c for c in df_raw.columns if "名称" in str(c) or "name" in str(c).lower()), None)
        if not code_col or not name_col:
            print(f"[securities] ERROR: missing code/name columns: {df_raw.columns.tolist()}")
            return

        df_norm = df_raw.rename(columns={code_col: "code", name_col: "name"}).copy()
        df_norm["ts_code"] = df_norm["code"].apply(infer_ts)
        keep = df_norm[["ts_code", "name"]]

    else:
        # tushare_basic
        if "ts_code" not in df_raw.columns or "name" not in df_raw.columns:
            print(f"[securities] ERROR: TuShare missing ts_code/name cols: {df_raw.columns.tolist()}")
            return
        keep = df_raw[["ts_code", "name"]]

    keep = keep[keep["ts_code"].str.contains(r"\.(?:SH|SZ|BJ)$", regex=True)]
    upsert_sqlite(keep, "securities", ["ts_code"])
    print(f"[securities] upserted {len(keep)} rows from {source}")

def _pick_code_column(df):
    """
    Detect a column that likely represents stock code.

    Core idea:
    - Handle inconsistent column names across data sources

    Typical usage:
    code_col = _pick_code_column(df)
    """
    candidates = [
        "代码", "品种代码", "证券代码", "成分券代码", "代码code", "代码/Code",
        "成分股代码", "con_code", "交易代码", "股票代码",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    for c in df.columns:
        if "代码" in str(c):
            return c

    return None


def fetch_index_members():
    """
    Fetch index constituents using multiple data sources.

    Core idea:
    - Try multiple AkShare APIs
    - Fallback to TuShare if needed
    - Normalize to index_code + ts_code

    Typical usage:
    fetch_index_members()
    """
    out = []

    for idx_code in CONFIG["INDEX_LIST"]:
        symbol = idx_code.split(".")[0]
        got = False

        # Try AkShare APIs
        for api_name in ["index_stock_cons_em", "index_stock_cons", "index_stock_cons_sina"]:
            func = getattr(ak, api_name, None)
            if func is None:
                continue

            df = safe_call(func, symbol=symbol, tries=3)

            if df is not None and not df.empty:
                code_col = _pick_code_column(df)

                if code_col:
                    df = df.rename(columns={code_col: "code"})
                    df["ts_code"] = df["code"].apply(infer_ts)

                    tmp = df[["ts_code"]].dropna().drop_duplicates()
                    tmp["index_code"] = idx_code
                    tmp["in_date"] = pd.NaT
                    tmp["out_date"] = pd.NaT

                    out.append(tmp[["index_code", "ts_code", "in_date", "out_date"]])

                    print(f"[index_members] via {api_name} {idx_code}: {len(tmp)}")
                    got = True
                    break

        if got:
            continue

        # TuShare fallback
        token = os.getenv("TUSHARE_TOKEN", "").strip()
        if token:
            try:
                import tushare as ts
                pro = ts.pro_api(token)

                df2 = pro.index_member(index_code=idx_code)

                if df2 is not None and not df2.empty:
                    df2 = df2.rename(columns={"con_code": "ts_code"})

                    df2["in_date"] = pd.to_datetime(df2.get("in_date"), errors="coerce")
                    df2["out_date"] = pd.to_datetime(df2.get("out_date"), errors="coerce")

                    keep = df2[["ts_code", "in_date", "out_date"]].dropna(subset=["ts_code"])
                    keep = keep.drop_duplicates(subset=["ts_code"])

                    keep["index_code"] = idx_code
                    out.append(keep[["index_code", "ts_code", "in_date", "out_date"]])

                    print(f"[index_members] via TuShare {idx_code}: {len(keep)}")
                    got = True

            except Exception as e:
                print(f"[index_members] TuShare failed {idx_code}: {e}")

        if not got:
            print(f"[index_members] WARN: failed {idx_code}")

        time.sleep(CONFIG["BASE_SLEEP"])

    if out:
        members = pd.concat(out, ignore_index=True).drop_duplicates()
        upsert_sqlite(members, "index_members", ["index_code", "ts_code"])
        print(f"[index_members] upsert -> {len(members)} rows")

  # =========================
# 5) MARKET DATA & CORP ACTIONS
# =========================

def _normalize_hist_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize daily OHLCV dataframe from different sources.

    Core idea:
    - Harmonize column names to: trade_date, open, high, low, close, vol(optional)
    - Parse trade_date as YYYY-MM-DD
    - Coerce numeric columns and remove duplicates

    Typical usage:
    df = _normalize_hist_df(df_raw)
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    # Common column mappings (Eastmoney / NetEase / generic)
    colmap_candidates = [
        {"日期": "trade_date", "开盘": "open", "最高": "high", "最低": "low", "收盘": "close", "成交量": "vol"},
        {"date": "trade_date", "open": "open", "high": "high", "low": "low", "close": "close", "volume": "vol"},
    ]
    for cmap in colmap_candidates:
        hit = [k for k in cmap.keys() if k in df.columns]
        if hit:
            df = df.rename(columns={k: cmap[k] for k in hit})

    # Ensure 'trade_date'
    if "trade_date" not in df.columns:
        for c in df.columns:
            s = str(c).lower()
            if ("日期" in str(c)) or s.startswith("date"):
                df = df.rename(columns={c: "trade_date"})
                break

    if "trade_date" not in df.columns:
        # Cannot normalize without date
        return pd.DataFrame()

    # Ensure volume column name (optional)
    if "vol" not in df.columns:
        if "成交量" in df.columns:
            df = df.rename(columns={"成交量": "vol"})
        elif "volume" in df.columns:
            df = df.rename(columns={"volume": "vol"})

    # Parse and standardize date
    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    df = df.dropna(subset=["trade_date"])

    # Coerce numeric columns (important for factor research)
    for c in ["open", "high", "low", "close", "vol"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Sort + de-duplicate by date
    df = df.sort_values("trade_date").drop_duplicates(subset=["trade_date"], keep="last").reset_index(drop=True)
    return df

def fetch_daily_k(
    ts_code: str,
    start: str = None,
    end: str = None,
    *,
    mode: str = "incremental",   # "incremental" or "full"
) -> int:
    """
    Fetch daily OHLCV (no adjustment) with incremental update.

    mode:
    - "incremental": only fetch missing dates after the last saved trade_date
    - "full": always fetch from start to end and overwrite

    Typical usage:
    fetch_daily_k("000001.SZ")                 # incremental (default)
    fetch_daily_k("000001.SZ", mode="full")    # full refresh
    """
    start = start or CONFIG["HIST_START"]
    end = end or (CONFIG["HIST_END"] or dt.date.today().strftime("%Y-%m-%d"))

    path = _parquet_path_for_symbol(ts_code)

    # Decide effective start date
    eff_start = start
    if mode == "incremental":
        last_date = _get_last_trade_date_from_parquet(path)
        if last_date:
            nxt = _next_day(last_date)
            if nxt is None:
                print(f"[daily_k] WARN: invalid last_date in parquet for {ts_code}, fallback to full range")
            else:
                eff_start = nxt

        # Already up-to-date
        if eff_start > end:
            print(f"[daily_k] {ts_code} up-to-date (last={last_date}, end={end})")
            return 0

    code = ts_code.split(".")[0]
    df, source = None, None

    # 1) Eastmoney hist
    df1 = safe_call(
    ak.stock_zh_a_hist,
    symbol=code,
    period="daily",
    adjust="",
    tries=6,
    sleep=1.2,
    jitter=1.5,
)
    if df1 is not None and not df1.empty:
        df, source = df1, "em_hist"

    # 2) NetEase hist (optional)
    if (df is None) or df.empty:
        func_163 = getattr(ak, "stock_zh_a_hist_163", None)
        if func_163 is not None:
            df2 = safe_call(func_163, symbol=code, period="daily", adjust="")
            if df2 is not None and not df2.empty:
                df, source = df2, "netease_hist"

    # 3) Fallback daily
    if (df is None) or df.empty:
        df3 = safe_call(ak.stock_zh_a_daily, symbol=code, adjust="")
        if df3 is not None and not df3.empty:
            df, source = df3, "em_daily"

    if df is None or df.empty:
        print(f"[daily_k] WARN: no data {ts_code}")
        return 0

    # Normalize and filter by date range (effective incremental window)
    df = _normalize_hist_df(df)
    if df is None or df.empty:
        print(f"[daily_k] WARN: normalization failed {ts_code} via {source}")
        return 0

    need_cols = ["trade_date", "open", "high", "low", "close"]
    if "vol" in df.columns:
        need_cols.append("vol")
    df = df[[c for c in need_cols if c in df.columns]].copy()

    df["ts_code"] = ts_code
    df = df[(df["trade_date"] >= eff_start) & (df["trade_date"] <= end)]

    if df.empty:
        print(f"[daily_k] {ts_code} no new rows (eff_start={eff_start}, end={end})")
        return 0

    # Merge with existing parquet (incremental) or overwrite (full)
    if mode == "incremental" and os.path.exists(path):
        try:
            old = pd.read_parquet(path)
            merged = pd.concat([old, df], ignore_index=True)
            merged = merged.drop_duplicates(subset=["trade_date"], keep="last")
            merged = merged.sort_values("trade_date").reset_index(drop=True)
            save_parquet(merged, path)
            print(f"[daily_k] {ts_code} +{len(df)} rows (merged={len(merged)}) via {source}")
            return int(len(df))
        except Exception as e:
            print(f"[daily_k] WARN: merge failed, overwrite parquet for {ts_code}: {e}")
            # fallback to overwrite with new chunk only (or you can choose full rebuild)
            save_parquet(df, path)
            print(f"[daily_k] {ts_code} wrote {len(df)} rows via {source}")
            return int(len(df))
    else:
        # full refresh or first time
        save_parquet(df, path)
        print(f"[daily_k] {ts_code} wrote {len(df)} rows via {source} (mode={mode})")
        return int(len(df))

def fetch_corp_actions_basic(ts_code: str):
    """
    Fetch basic corporate actions (dividends / splits etc.) and upsert into SQLite.

    Core idea:
    - Use a single canonical schema: ex_date + cash dividend + split ratios
    - Store in SQLite as metadata table

    Typical usage:
    fetch_corp_actions_basic("000001.SZ")
    """
    code = ts_code.split(".")[0]
    df = safe_call(ak.stock_history_dividend_detail, symbol=code, tries=3)

    if df is None or df.empty:
        print(f"[corp_actions] WARN: empty {ts_code}")
        return

    df = df.copy()

    # Flexible column picking (AkShare tables may differ)
    ex_date_series = None
    if "除权除息日" in df.columns:
        ex_date_series = df["除权除息日"]
    elif "分红实施公告日" in df.columns:
        ex_date_series = df["分红实施公告日"]

    div_cash_series = None
    if "每股股息(税前)" in df.columns:
        div_cash_series = df["每股股息(税前)"]
    elif "分红金额(税前)" in df.columns:
        div_cash_series = df["分红金额(税前)"]

    keep = pd.DataFrame({
        "ex_date": pd.to_datetime(ex_date_series, errors="coerce"),
        "div_cash": pd.to_numeric(div_cash_series, errors="coerce") if div_cash_series is not None else pd.NA,
        "bonus_share": pd.to_numeric(df.get("送股比例"), errors="coerce"),
        "transfer_share": pd.to_numeric(df.get("转增比例"), errors="coerce"),
        "rights_issue_ratio": pd.to_numeric(df.get("配股比例"), errors="coerce"),
        "rights_price": pd.to_numeric(df.get("配股价格"), errors="coerce"),
    })

    keep["ts_code"] = ts_code
    keep = keep.dropna(subset=["ex_date"])

    if keep.empty:
        print(f"[corp_actions] WARN: no valid ex_date rows {ts_code}")
        return

    upsert_sqlite(keep, "corp_actions", ["ts_code", "ex_date"])
    print(f"[corp_actions] upserted {len(keep)} rows for {ts_code}")

def batch_fetch_corp_actions(sample_n: int = 40):
    """
    Batch fetch corporate actions for a subset of symbols.

    Typical usage:
    batch_fetch_corp_actions(40)
    """
    try:
        secs = pd.read_sql_table("securities", con=engine)
    except Exception as e:
        print(f"[corp_actions] ERR: cannot read securities table: {e}")
        return

    if secs is None or secs.empty or "ts_code" not in secs.columns:
        print("[corp_actions] ERR: securities table empty or missing ts_code")
        return

    n = min(int(sample_n), len(secs))
    selected = secs.head(n)

    ok, fail = 0, 0
    for ts in selected["ts_code"]:
        try:
            fetch_corp_actions_basic(ts)
            ok += 1
        except Exception as e:
            fail += 1
            print(f"[corp_actions] soft-fail {ts}: {e}")

        time.sleep(CONFIG["BASE_SLEEP"])

    print(f"[corp_actions] summary -> ok:{ok}, fail:{fail}, symbols:{len(selected)}")

    return {
    "rows": ok,
    "symbols": len(selected),
    "fail": fail,
}

def batch_fetch_daily(sample_n: int = None, *, mode: str = "incremental"):
    """
    Batch daily bars update.
    mode: "incremental" (recommended) or "full"
    """
    if sample_n is None:
        sample_n = CONFIG.get("SAMPLE_N", None)

    try:
        secs = pd.read_sql_table("securities", con=engine)
    except Exception as e:
        print(f"[daily_k] ERR: cannot read securities table: {e}")
        return

    if secs is None or secs.empty or "ts_code" not in secs.columns:
        print("[daily_k] ERR: securities table empty or missing ts_code")
        return

    if sample_n is None:
        selected = secs  # full universe
    else:
        n = min(int(sample_n), len(secs))
        selected = secs.sample(n, random_state=42)

    total_new, ok, fail, zero = 0, 0, 0, 0

    for ts_code in selected["ts_code"]:
        try:
            n_new = fetch_daily_k(ts_code, mode=mode)
            total_new += (n_new or 0)
            if (n_new or 0) > 0:
                ok += 1
            else:
                zero += 1
        except Exception as e:
            fail += 1
            print(f"[daily_k] soft-fail {ts_code}: {e}")

        time.sleep(CONFIG["BASE_SLEEP"])

    print(f"[daily_k] summary -> ok:{ok}, zero:{zero}, fail:{fail}, new_rows:{total_new}, symbols:{len(selected)}")

    return {
    "rows": total_new,
    "symbols": len(selected),
    "ok": ok,
    "zero": zero,
    "fail": fail,
}

# =========================
# 6)STANDARDIZED LAYER
# =========================
def _standardized_daily_bars_root() -> str:
    return f'{CONFIG["DATA_DIR"]}/standardized/market/daily_bars'

def reset_standardized_daily_bars():
    """
    Remove entire standardized daily_bars dataset and recreate it.
    Best for Colab/dev stage.
    """
    root = _standardized_daily_bars_root()
    if os.path.exists(root):
        shutil.rmtree(root)
    os.makedirs(root, exist_ok=True)
    print(f"[standardized_daily_bars] reset dataset at {root}")

def write_partitioned_by_trade_date(df: pd.DataFrame, root_path: str):
    """
    Write dataframe into parquet partitions:
    root_path/trade_date=YYYY-MM-DD/data.parquet
    """
    if df is None or df.empty:
        print("[partition_write] WARN: empty dataframe")
        return

    os.makedirs(root_path, exist_ok=True)

    for trade_date, part in df.groupby("trade_date"):
        part_dir = os.path.join(root_path, f"trade_date={trade_date}")
        os.makedirs(part_dir, exist_ok=True)

        file_path = os.path.join(part_dir, "data.parquet")
        part = part.sort_values("ts_code").reset_index(drop=True)
        part.to_parquet(file_path, index=False)

    print(f"[partition_write] wrote {df['trade_date'].nunique()} partitions")

def rebuild_standardized_daily_bars():
    """
    Rebuild standardized daily_bars dataset from raw per-symbol parquet files.

    Flow:
    raw daily_by_symbol -> concat -> normalize -> validate -> reset -> partition write
    """
    src_dir = f'{CONFIG["DATA_DIR"]}/raw/market/daily_by_symbol'
    out_dir = _standardized_daily_bars_root()

    files = glob.glob(os.path.join(src_dir, "*.parquet"))
    if not files:
        print("[standardized_daily_bars] WARN: no raw parquet files found")
        return

    dfs = []
    for fp in files:
        try:
            df = pd.read_parquet(fp)
            if df is None or df.empty:
                continue

            # normalize again for safety
            df = _normalize_hist_df(df)
            if df is None or df.empty:
                continue

            # enforce columns
            need_cols = ["trade_date", "open", "high", "low", "close"]
            if "vol" in df.columns:
                need_cols.append("vol")

            if "ts_code" not in df.columns:
                # infer from filename if missing
                fname = os.path.basename(fp).replace(".parquet", "")
                parts = fname.rsplit("_", 1)
                ts_code = f"{parts[0]}.{parts[1]}" if len(parts) == 2 else fname
                df["ts_code"] = ts_code

            df = df[[c for c in ["ts_code"] + need_cols if c in df.columns]].copy()
            dfs.append(df)

        except Exception as e:
            print(f"[standardized_daily_bars] WARN: failed reading {fp}: {e}")

    if not dfs:
        print("[standardized_daily_bars] WARN: no valid raw data")
        return

    full = pd.concat(dfs, ignore_index=True)

    # standardize types / sort / dedup
    full["trade_date"] = pd.to_datetime(full["trade_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    for c in ["open", "high", "low", "close", "vol"]:
        if c in full.columns:
            full[c] = pd.to_numeric(full[c], errors="coerce")

    full = full.dropna(subset=["ts_code", "trade_date", "open", "high", "low", "close"])
    full = full.drop_duplicates(subset=["ts_code", "trade_date"], keep="last")
    full = full.sort_values(["trade_date", "ts_code"]).reset_index(drop=True)

    if full.empty:
        print("[standardized_daily_bars] WARN: full dataset empty after cleaning")
        return

    # quality check
    issues = validate_daily_bars(full)
    if issues is not None and not issues.empty:
        print("[standardized_daily_bars] quality issues found:")
        print(issues)
    else:
        print("[standardized_daily_bars] quality checks passed")

    write_quality_report("standardized_daily_bars", issues)

    # rebuild dataset
    reset_standardized_daily_bars()
    write_partitioned_by_trade_date(full, out_dir)

    update_dataset_registry(
        dataset_name="standardized_daily_bars",
        layer="standardized",
        path=out_dir,
        row_count=len(full),
        partition_count=df["trade_date"].nunique(),
        schema_version="v1",
)

    print(
        f"[standardized_daily_bars] rebuilt rows={len(full)} "
        f"partitions={full['trade_date'].nunique()} symbols={full['ts_code'].nunique()}"
    )

    return {
    "rows": len(full),
    "partitions": int(full["trade_date"].nunique()),
    "symbols": int(full["ts_code"].nunique()),
}

# =========================
# 7) FEATURE LAYER
# =========================
def load_standardized_daily_bars() -> pd.DataFrame:
    root = _standardized_daily_bars_root()
    files = glob.glob(os.path.join(root, "trade_date=*/data.parquet"))

    if not files:
        print("[load_standardized] WARN: no partition files found")
        return pd.DataFrame()

    dfs = []
    for fp in files:
        try:
            df = pd.read_parquet(fp)
            if df is not None and not df.empty:
                dfs.append(df)
        except Exception as e:
            print(f"[load_standardized] WARN: failed reading {fp}: {e}")

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)
    out = out.sort_values(["ts_code", "trade_date"]).reset_index(drop=True)
    return out

def build_market_features():
    df = load_standardized_daily_bars()
    if df is None or df.empty:
        print("[market_features] WARN: empty standardized daily bars")
        return

    df = df.copy()

    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
    df = df.dropna(subset=["trade_date"])
    df = df.sort_values(["ts_code", "trade_date"]).reset_index(drop=True)

    g = df.groupby("ts_code", group_keys=False)

    df["ret_1d"] = g["close"].pct_change(1)
    df["ret_5d"] = g["close"].pct_change(5)
    df["ret_20d"] = g["close"].pct_change(20)

    df["ma_5"] = g["close"].transform(lambda s: s.rolling(5, min_periods=5).mean())
    df["ma_20"] = g["close"].transform(lambda s: s.rolling(20, min_periods=20).mean())

    df["ma_ratio_20"] = df["close"] / df["ma_20"] - 1
    df["ma_ratio_20"] = df["ma_ratio_20"].replace([np.inf, -np.inf], np.nan)

    df["volatility_20d"] = g["ret_1d"].transform(
        lambda s: s.rolling(20, min_periods=20).std()
    )

    df = df.drop_duplicates(subset=["ts_code", "trade_date"], keep="last")

    df = df.sort_values(["trade_date", "ts_code"]).reset_index(drop=True)

    df["trade_date"] = df["trade_date"].dt.strftime("%Y-%m-%d")

    issues = validate_market_features(df)
    if issues is not None and not issues.empty:
        print("[market_features] quality issues found:")
        print(issues)
    else:
        print("[market_features] quality checks passed")

    write_quality_report("market_features", issues)

    out_dir = f'{CONFIG["DATA_DIR"]}/feature/market_features'
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    write_partitioned_by_trade_date(df, out_dir)

    update_dataset_registry(
        dataset_name="market_features",
        layer="feature",
        path=out_dir,
        row_count=len(df),
        partition_count=df["trade_date"].nunique(),
        schema_version="v1",
)

    print(
        f"[market_features] wrote rows={len(df)} "
        f"partitions={df['trade_date'].nunique()} "
        f"symbols={df['ts_code'].nunique()}"
    )

    return {
    "rows": len(df),
    "partitions": int(df["trade_date"].nunique()),
    "symbols": int(df["ts_code"].nunique()),
}

# =========================
# 8) ENTRYPOINT
# =========================

def _run_step(name: str, enabled: bool, fn, *args, **kwargs):
    if not enabled:
        print(f"[{name}] skipped")
        return

    run_id = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    start_dt = dt.datetime.now()
    start_time = start_dt.strftime("%Y-%m-%d %H:%M:%S")

    print(f"[{name}] start")
    try:
        result = fn(*args, **kwargs)

        end_dt = dt.datetime.now()
        end_time = end_dt.strftime("%Y-%m-%d %H:%M:%S")
        duration_seconds = (end_dt - start_dt).total_seconds()

        rows = None
        if isinstance(result, dict):
            rows = result.get("rows")

        append_pipeline_run(
            run_id=run_id,
            step_name=name,
            start_time=start_time,
            end_time=end_time,
            status="success",
            message="",
            rows=rows,
            duration_seconds=duration_seconds,
        )

        print(f"[{name}] done")
    except Exception as e:
        end_dt = dt.datetime.now()
        end_time = end_dt.strftime("%Y-%m-%d %H:%M:%S")
        duration_seconds = (end_dt - start_dt).total_seconds()

        append_pipeline_run(
            run_id=run_id,
            step_name=name,
            start_time=start_time,
            end_time=end_time,
            status="failed",
            message=str(e),
            rows=None,
            duration_seconds=duration_seconds,
        )

        print(f"[{name}] soft-fail: {e}")

def main(*, daily_mode: str = "incremental"):
    """
    Main pipeline entry.
    """
    ensure_dirs()

    runs = CONFIG.get("RUNS", {})

    _run_step("trading_calendar", runs.get("trading_calendar", False), fetch_trading_calendar)
    _run_step("securities_list",  runs.get("securities_list", False),  fetch_securities_list)
    _run_step("index_members",    runs.get("index_members", False),    fetch_index_members)

    _run_step(
        "daily_bars",
        runs.get("daily_bars", False),
        batch_fetch_daily,
        sample_n=CONFIG.get("SAMPLE_N", None),
        mode=daily_mode,
    )

    _run_step(
        "standardized_daily_bars",
        runs.get("standardized_daily_bars", False),
        rebuild_standardized_daily_bars,
    )

    _run_step(
        "market_features",
        runs.get("market_features", False),
        build_market_features,
)

    _run_step(
        "corp_actions",
        runs.get("corp_actions", False),
        batch_fetch_corp_actions,
        sample_n=CONFIG.get("CORP_ACTIONS_SAMPLE_N", 40),
    )

    print("[main] finished")

# =========================
# 8) EXECUTION(First time)
# =========================
FIRST_RUN = False
main(daily_mode="full" if FIRST_RUN else "incremental")

In [ ]:
# =========================
# 9) INSPECT
# =========================
def inspect_standardized_daily_bars(n_parts=3, n_rows=5):
    root = _standardized_daily_bars_root()
    parts = sorted(glob.glob(os.path.join(root, "trade_date=*")))
    print(f"[inspect] total partitions = {len(parts)}")

    for p in parts[:n_parts]:
        fp = os.path.join(p, "data.parquet")
        if not os.path.exists(fp):
            print(f"[inspect] missing file: {fp}")
            continue

        df = pd.read_parquet(fp)
        print(f"\n[{os.path.basename(p)}] rows={len(df)}")
        print(df.head(n_rows))

def inspect_market_features(n_parts=3, n_rows=5, start_idx=0):
    root = f'{CONFIG["DATA_DIR"]}/feature/market_features'
    parts = sorted(glob.glob(os.path.join(root, "trade_date=*")))
    print(f"[inspect_features] total partitions = {len(parts)}")

    for p in parts[start_idx:start_idx + n_parts]:
        fp = os.path.join(p, "data.parquet")
        if not os.path.exists(fp):
            print(f"[inspect_features] missing file: {fp}")
            continue

        df = pd.read_parquet(fp)
        print(f"\n[{os.path.basename(p)}] rows={len(df)}")
        print(df.head(n_rows))

def show_dataset_registry():
    try:
        df = pd.read_sql_table("dataset_registry", con=engine)
        if df.empty:
            print("[dataset_registry] empty")
            return
        print(df.sort_values(["layer", "dataset_name"]).reset_index(drop=True))
    except Exception as e:
        print(f"[dataset_registry] ERR: {e}")

def show_pipeline_runs(n=20):
    try:
        df = pd.read_sql_table("pipeline_runs", con=engine)
        if df.empty:
            print("[pipeline_runs] empty")
            return

        df = df.sort_values(["start_time", "step_name"], ascending=[False, True]).reset_index(drop=True)
        print(df.head(n))
    except Exception as e:
        print(f"[pipeline_runs] ERR: {e}")

def show_data_quality_reports(n=20):
    try:
        df = pd.read_sql_table("data_quality_reports", con=engine)
        if df.empty:
            print("[data_quality_reports] empty")
            return
        print(df.sort_values(["checked_at", "dataset_name"], ascending=[False, True]).head(n).reset_index(drop=True))
    except Exception as e:
        print(f"[data_quality_reports] ERR: {e}")

inspect_standardized_daily_bars()

inspect_market_features(start_idx=30)

show_dataset_registry()

show_pipeline_runs()

show_data_quality_reports()